# Historical options exploration

This offline example treats the option chain as observed provider data. Calculations that need spot receive an explicit underlying price; no pricing model or Greek calculation is implied.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt

from persistra.analysis import (
    chain_summary,
    filter_chain,
    implied_volatility_smile,
    intrinsic_value,
    moneyness,
    option_relative_spread,
)
from persistra.data import DuckDBStore, synthetic
from persistra.viz import (
    plot_greek_profile,
    plot_implied_volatility_smile,
    plot_implied_volatility_surface,
    plot_option_chain_prices,
    plot_option_volume_open_interest,
)

In [ ]:
chain = synthetic.option_chain("DEMO")
expiration = chain.contracts["expiration"].dt.date.min()
calls = filter_chain(chain, expiration=expiration, option_type="call")
summary = chain_summary(chain)
smile = implied_volatility_smile(chain, expiration=expiration)
summary

In [ ]:
underlying_price = 100.0
relative_strike = moneyness(calls, underlying_price=underlying_price)
payoff_floor = intrinsic_value(calls, underlying_price=underlying_price)
observed_spreads = option_relative_spread(calls)
assert len(relative_strike) == len(payoff_floor) == len(calls.contracts)
observed_spreads

In [ ]:
temporary_directory = TemporaryDirectory()
store_path = Path(temporary_directory.name) / "options.duckdb"
with DuckDBStore.create(store_path) as store:
    store.save(chain)
    restored = store.load_options(chain.underlying_instrument_id, chain.chain_date)
assert restored is not None and len(restored.contracts) == len(chain.contracts)

In [ ]:
plot_option_chain_prices(chain)
plot_option_volume_open_interest(chain)
plot_implied_volatility_smile(chain, expiration=expiration)
plot_implied_volatility_surface(chain)
plot_greek_profile(chain, "delta", expiration=expiration)
plt.close("all")
temporary_directory.cleanup()